# Time Series Forecast Model B Workflow


In [2]:
from time_series_forecast import TimeSeriesForecastConfig, TimeSeriesForecastPipeline
import pandas as pd

config = TimeSeriesForecastConfig(
    data_path="data/ChargePointData2011-2020-ManullyPreCleaned3.csv",
    model_dir="model_B",
    test_size=0.2,
    time_steps=12,
)

pipeline = TimeSeriesForecastPipeline(config)

## 1. Build the Final Dataset

In [3]:
pipeline.load_data()
final_df = pipeline.build_dataset(available=False)

final_df.head()

Fetching Palo Alto weather data...
Final dataset shape: (259415, 16)
Duration target summary:
count    259415.000000
mean       8952.554960
std        7965.581056
min          60.000000
25%        4215.000000
50%        7521.000000
75%       11124.000000
max      412341.000000
Name: Total_Duration_Secs, dtype: float64


,MAC Address,Total_Duration_Secs,temp,precip,clouds,Minute,Hour,DayOfWeek,Month,IsWeekend,IsHoliday,Hour_Sin,Hour_Cos,Month_Sin,Month_Cos,Plug Type Encoded
0,000D:6F00:015A:9D76,11012.0,18.7,0.0,2,17,20,4,7,0,0,-0.866025,5.000000e-01,-0.5,-0.866025,0
1,000D:6F00:015A:9D76,126.0,15.9,0.0,77,0,0,5,7,1,0,0.000000,1.000000e+00,-0.5,-0.866025,0
2,000D:6F00:015A:9D76,15452.0,15.4,0.0,97,16,8,5,7,1,0,0.866025,-5.000000e-01,-0.5,-0.866025,0
3,000D:6F00:015A:9D76,7404.0,24.1,0.0,2,51,14,5,7,1,0,-0.500000,-8.660254e-01,-0.5,-0.866025,0
4,000D:6F00:015A:9D76,4284.0,22.2,0.0,6,51,18,5,7,1,0,-1.000000,-1.836970e-16,-0.5,-0.866025,0


In [4]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 259415 entries, 0 to 259414
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   MAC Address          259415 non-null  str    
 1   Total_Duration_Secs  259415 non-null  float64
 2   temp                 259415 non-null  float64
 3   precip               259415 non-null  float64
 4   clouds               259415 non-null  int64  
 5   Minute               259415 non-null  int32  
 6   Hour                 259415 non-null  int32  
 7   DayOfWeek            259415 non-null  int32  
 8   Month                259415 non-null  int32  
 9   IsWeekend            259415 non-null  int64  
 10  IsHoliday            259415 non-null  int64  
 11  Hour_Sin             259415 non-null  float64
 12  Hour_Cos             259415 non-null  float64
 13  Month_Sin            259415 non-null  float64
 14  Month_Cos            259415 non-null  float64
 15  Plug Type Encoded    259415 

## 2. Split the Label

In [5]:
split_idx = int(len(final_df) * 0.8)
train_df = final_df.iloc[:split_idx]
test_df = final_df.iloc[split_idx:]
train_df

,MAC Address,Total_Duration_Secs,temp,precip,clouds,Minute,Hour,DayOfWeek,Month,IsWeekend,IsHoliday,Hour_Sin,Hour_Cos,Month_Sin,Month_Cos,Plug Type Encoded
0,000D:6F00:015A:9D76,11012.0,18.7,0.0,2,17,20,4,7,0,0,-0.866025,5.000000e-01,-0.5,-0.866025,0
1,000D:6F00:015A:9D76,126.0,15.9,0.0,77,0,0,5,7,1,0,0.000000,1.000000e+00,-0.5,-0.866025,0
2,000D:6F00:015A:9D76,15452.0,15.4,0.0,97,16,8,5,7,1,0,0.866025,-5.000000e-01,-0.5,-0.866025,0
3,000D:6F00:015A:9D76,7404.0,24.1,0.0,2,51,14,5,7,1,0,-0.500000,-8.660254e-01,-0.5,-0.866025,0
4,000D:6F00:015A:9D76,4284.0,22.2,0.0,6,51,18,5,7,1,0,-1.000000,-1.836970e-16,-0.5,-0.866025,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207527,0024:B100:0002:1323,6693.0,20.9,0.0,10,59,13,5,5,1,0,-0.258819,-9.659258e-01,0.5,-0.866025,0
207528,0024:B100:0002:6D37,7130.0,22.2,0.0,0,0,14,5,5,1,0,-0.500000,-8.660254e-01,0.5,-0.866025,0
207529,000D:6F00:04B8:A128,6847.0,22.2,0.0,0,2,14,5,5,1,0,-0.500000,-8.660254e-01,0.5,-0.866025,0
207530,0024:B100:0003:4209,4872.0,22.2,0.0,0,10,14,5,5,1,0,-0.500000,-8.660254e-01,0.5,-0.866025,0


## 3. Pipeline the Data

In [6]:
train_df_label_duration = train_df['Total_Duration_Secs'].copy()
train_df_data = train_df.drop(['Total_Duration_Secs'], axis = 1)
test_df_label_duration = test_df['Total_Duration_Secs'].copy()
test_df_data = test_df.drop(['Total_Duration_Secs'], axis = 1)
print("Train data shape:", train_df_data.shape)
print("Duration label shape:", train_df_label_duration.shape)
print("Test data shape:", test_df_data.shape)
print("Test duration label shape:", test_df_label_duration.shape)

Train data shape: (207532, 15)
Duration label shape: (207532,)
Test data shape: (51883, 15)
Test duration label shape: (51883,)


In [7]:
train_df_data

,MAC Address,temp,precip,clouds,Minute,Hour,DayOfWeek,Month,IsWeekend,IsHoliday,Hour_Sin,Hour_Cos,Month_Sin,Month_Cos,Plug Type Encoded
0,000D:6F00:015A:9D76,18.7,0.0,2,17,20,4,7,0,0,-0.866025,5.000000e-01,-0.5,-0.866025,0
1,000D:6F00:015A:9D76,15.9,0.0,77,0,0,5,7,1,0,0.000000,1.000000e+00,-0.5,-0.866025,0
2,000D:6F00:015A:9D76,15.4,0.0,97,16,8,5,7,1,0,0.866025,-5.000000e-01,-0.5,-0.866025,0
3,000D:6F00:015A:9D76,24.1,0.0,2,51,14,5,7,1,0,-0.500000,-8.660254e-01,-0.5,-0.866025,0
4,000D:6F00:015A:9D76,22.2,0.0,6,51,18,5,7,1,0,-1.000000,-1.836970e-16,-0.5,-0.866025,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207527,0024:B100:0002:1323,20.9,0.0,10,59,13,5,5,1,0,-0.258819,-9.659258e-01,0.5,-0.866025,0
207528,0024:B100:0002:6D37,22.2,0.0,0,0,14,5,5,1,0,-0.500000,-8.660254e-01,0.5,-0.866025,0
207529,000D:6F00:04B8:A128,22.2,0.0,0,2,14,5,5,1,0,-0.500000,-8.660254e-01,0.5,-0.866025,0
207530,0024:B100:0003:4209,22.2,0.0,0,10,14,5,5,1,0,-0.500000,-8.660254e-01,0.5,-0.866025,0


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from category_encoders import BinaryEncoder

num_attribs = list(train_df_data.select_dtypes(include="number").columns)
cat_attribs = ['MAC Address']
# Using Binary Encoding keeps the feature space small enough for LSTM
# and provides enough structure for XGBoost.
preprocessor = ColumnTransformer(
    transformers=[
        ('mac_bits', BinaryEncoder(), ['MAC Address']),
        ('std_scaler', StandardScaler(), num_attribs)
    ],
    remainder='passthrough' 
)
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
])

train_df_prepared = full_pipeline.fit_transform(train_df_data, train_df_label_duration)
feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()
train_df_prepared = pd.DataFrame(train_df_prepared, columns=feature_names, index=train_df_data.index)
train_df_prepared.head()

,mac_bits__MAC Address_0,mac_bits__MAC Address_1,mac_bits__MAC Address_2,mac_bits__MAC Address_3,mac_bits__MAC Address_4,mac_bits__MAC Address_5,mac_bits__MAC Address_6,std_scaler__temp,std_scaler__precip,std_scaler__clouds,...,std_scaler__Hour,std_scaler__DayOfWeek,std_scaler__Month,std_scaler__IsWeekend,std_scaler__IsHoliday,std_scaler__Hour_Sin,std_scaler__Hour_Cos,std_scaler__Month_Sin,std_scaler__Month_Cos,std_scaler__Plug Type Encoded
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.315914,-0.178408,-1.021264,...,1.540475,0.602278,0.206339,-0.554484,-0.135783,-1.018474,1.849723,-0.765885,-1.237721,-0.151759
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.168954,-0.178408,0.780855,...,-2.970230,1.123632,0.206339,1.803477,-0.135783,0.188940,2.822627,-0.765885,-1.237721,-0.151759
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.255537,-0.178408,1.261421,...,-1.165948,1.123632,0.206339,1.803477,-0.135783,1.396353,-0.096084,-0.765885,-1.237721,-0.151759
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.251016,-0.178408,-1.021264,...,0.187263,1.123632,0.206339,1.803477,-0.135783,-0.508161,-0.808299,-0.765885,-1.237721,-0.151759
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.921998,-0.178408,-0.925151,...,1.089404,1.123632,0.206339,1.803477,-0.135783,-1.205261,0.876819,-0.765885,-1.237721,-0.151759


In [50]:
import joblib

# Save the entire preprocessing logic
joblib.dump(full_pipeline, 'preprocessor_pipeline.joblib')

['preprocessor_pipeline.joblib']

## 4. Train ML Model

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

lr = LinearRegression()
lr_scores = cross_val_score(lr, train_df_prepared, train_df_label_duration, scoring="neg_mean_absolute_error", cv=5)
print("Scores:", lr_scores)

Scores: [-5316.17605222 -5511.88004384 -5598.71331125 -4571.36768193
 -4379.3087996 ]


In [10]:
import xgboost as xgb

xgb_rg = xgb.XGBRegressor()
xgb_rg.fit(train_df_prepared, train_df_label_duration)
xgb_rg_scores = cross_val_score(xgb_rg, train_df_prepared, train_df_label_duration, scoring="neg_mean_absolute_error", cv=5, n_jobs=-1)
print("Scores:", xgb_rg_scores)
# Check recall and precision as well, SAVE THE MODEL

Scores: [-5049.70501758 -5343.89806201 -5396.45593879 -4278.52970884
 -4036.38306215]


In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

param_grid_lr = {
    "fit_intercept": [True, False],
    "positive": [True, False] # Forces coefficients to be positive if True
}

param_grid_dt = {
    "criterion": ["squared_error", "friedman_mse"],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10]
}

param_grid_rf = {
    "n_estimators": [100, 200],
    "criterion": ["squared_error"],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0]
}

models = [
    ("LinearRegression", LinearRegression(), param_grid_lr),
    ("DecisionTree", DecisionTreeRegressor(random_state=42), param_grid_dt),
    ("RandomForest", RandomForestRegressor(random_state=42), param_grid_rf),
    ("XGBRegressor", xgb.XGBRegressor(eval_metric="rmse", random_state=42), param_grid_xgb),
]

best_results = {}
saved = True
if not saved:
    for name, estimator, param_grid in models:
        grid = GridSearchCV(estimator, param_grid, scoring="neg_mean_absolute_error", cv=5, n_jobs=-1, verbose=1)
        grid.fit(train_df_prepared, train_df_label_duration)
        best_results[name] = {
            "best_score": grid.best_score_,
            "best_params": grid.best_params_,
            "best_estimator": grid.best_estimator_
        }
        print(f"{name}: best_score={grid.best_score_:.4f}")
        print("best_params=", grid.best_params_)
        print()

    best_results
# Random forest took the longest time


In [13]:
import joblib
from pathlib import Path

model_dir = Path(config.model_dir)
model_dir.mkdir(exist_ok=True)

if not saved:
    joblib.dump(best_results['XGBRegressor']['best_estimator'], model_dir / 'best_xgb_model.pkl')
    joblib.dump(best_results['XGBRegressor']['best_params'], model_dir / 'best_xgb_params.pkl')

    joblib.dump(best_results['RandomForest']['best_estimator'], model_dir / 'best_rf_model.pkl')
    joblib.dump(best_results['RandomForest']['best_params'], model_dir / 'best_rf_params.pkl')

    joblib.dump(best_results['DecisionTree']['best_estimator'], model_dir / 'best_dt_model.pkl')
    joblib.dump(best_results['DecisionTree']['best_params'], model_dir / 'best_dt_params.pkl')

    joblib.dump(best_results['LinearRegression']['best_estimator'], model_dir / 'best_lr_model.pkl')
    joblib.dump(best_results['LinearRegression']['best_params'], model_dir / 'best_lr_params.pkl')

best_xgb_model = joblib.load(model_dir / 'best_xgb_model.pkl')
best_xgb_params = joblib.load(model_dir / 'best_xgb_params.pkl')

best_rf_model = joblib.load(model_dir / 'best_rf_model.pkl')
best_rf_params = joblib.load(model_dir / 'best_rf_params.pkl')

best_dt_model = joblib.load(model_dir / 'best_dt_model.pkl')
best_dt_params = joblib.load(model_dir / 'best_dt_params.pkl')

best_lr_model = joblib.load(model_dir / 'best_lr_model.pkl')
best_lr_params = joblib.load(model_dir / 'best_lr_params.pkl')

In [14]:
# Assemble
from sklearn.ensemble import VotingRegressor

xgb_best = best_xgb_model
lr_best = best_lr_model
rf_best = best_rf_model
dc_best= best_dt_model

voting_regressor = VotingRegressor(
    estimators=[('lr', lr_best), ('rf', rf_best), ('dc', dc_best), ('xgb', xgb_best)],
    n_jobs=-1
)
voting_regressor.fit(train_df_prepared, train_df_label_duration)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lr', ...), ('rf', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'


# Test Set

In [31]:
final_test_df_data = full_pipeline.transform(test_df_data)
final_test_df_data = pd.DataFrame(final_test_df_data, columns=feature_names, index=test_df_data.index)

In [32]:
final_test_df_data

,mac_bits__MAC Address_0,mac_bits__MAC Address_1,mac_bits__MAC Address_2,mac_bits__MAC Address_3,mac_bits__MAC Address_4,mac_bits__MAC Address_5,mac_bits__MAC Address_6,std_scaler__temp,std_scaler__precip,std_scaler__clouds,...,std_scaler__Hour,std_scaler__DayOfWeek,std_scaler__Month,std_scaler__IsWeekend,std_scaler__IsHoliday,std_scaler__Hour_Sin,std_scaler__Hour_Cos,std_scaler__Month_Sin,std_scaler__Month_Cos,std_scaler__Plug Type Encoded
207532,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207533,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207534,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207535,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207536,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259410,0.0,1.0,1.0,1.0,0.0,0.0,1.0,-0.584554,-0.178408,0.132092,...,0.638334,0.080924,1.651667,-0.554484,-0.135783,-1.018474,-0.096084,-0.065080,1.431270,-0.151759
259411,0.0,1.0,1.0,0.0,0.0,1.0,0.0,-0.584554,-0.178408,0.132092,...,0.638334,0.080924,1.651667,-0.554484,-0.135783,-1.018474,-0.096084,-0.065080,1.431270,-0.151759
259412,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.601871,-0.178408,0.132092,...,0.863869,0.080924,1.651667,-0.554484,-0.135783,-1.157755,0.373207,-0.065080,1.431270,-0.151759
259413,0.0,1.0,1.0,0.0,0.0,1.0,0.0,-0.601871,-0.178408,0.132092,...,0.863869,0.080924,1.651667,-0.554484,-0.135783,-1.157755,0.373207,-0.065080,1.431270,-0.151759


In [17]:
from sklearn.metrics import mean_absolute_error

for reg in (lr_best, rf_best, dc_best, xgb_best, voting_regressor):
    y_pred = reg.predict(final_test_df_data)
    print(reg.__class__.__name__, mean_absolute_error(test_df_label_duration, y_pred))

d:\University\Final_Project\Coding\AI\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
d:\University\Final_Project\Coding\AI\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


LinearRegression 4770.0130298744525
RandomForestRegressor 4409.253281494323
DecisionTreeRegressor 4530.713506632372
XGBRegressor 4346.843584052323


d:\University\Final_Project\Coding\AI\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(


VotingRegressor 4470.694897920758


## 5. Train LSTM Model

In [18]:
train_df_prepared = pd.DataFrame(train_df_prepared)
train_df_prepared

,mac_bits__MAC Address_0,mac_bits__MAC Address_1,mac_bits__MAC Address_2,mac_bits__MAC Address_3,mac_bits__MAC Address_4,mac_bits__MAC Address_5,mac_bits__MAC Address_6,std_scaler__temp,std_scaler__precip,std_scaler__clouds,...,std_scaler__Hour,std_scaler__DayOfWeek,std_scaler__Month,std_scaler__IsWeekend,std_scaler__IsHoliday,std_scaler__Hour_Sin,std_scaler__Hour_Cos,std_scaler__Month_Sin,std_scaler__Month_Cos,std_scaler__Plug Type Encoded
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.315914,-0.178408,-1.021264,...,1.540475,0.602278,0.206339,-0.554484,-0.135783,-1.018474,1.849723,-0.765885,-1.237721,-0.151759
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.168954,-0.178408,0.780855,...,-2.970230,1.123632,0.206339,1.803477,-0.135783,0.188940,2.822627,-0.765885,-1.237721,-0.151759
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.255537,-0.178408,1.261421,...,-1.165948,1.123632,0.206339,1.803477,-0.135783,1.396353,-0.096084,-0.765885,-1.237721,-0.151759
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.251016,-0.178408,-1.021264,...,0.187263,1.123632,0.206339,1.803477,-0.135783,-0.508161,-0.808299,-0.765885,-1.237721,-0.151759
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.921998,-0.178408,-0.925151,...,1.089404,1.123632,0.206339,1.803477,-0.135783,-1.205261,0.876819,-0.765885,-1.237721,-0.151759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207527,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.696881,-0.178408,-0.829038,...,-0.038272,1.123632,-0.371793,1.803477,-0.135783,-0.171906,-1.002686,0.635725,-1.237721,-0.151759
207528,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207529,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207530,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759


In [38]:
import numpy as np

# 1. Identify your Binary Encoded columns and other features
# This assumes BinaryEncoder prefixed them with 'MAC Address_'
features = [col for col in train_df_prepared.columns]

# 2. IMPORTANT: We need the ORIGINAL 'MAC Address' to group correctly.
# If you dropped it during transform, we pull it from the original train_df_data.
temp_df = pd.concat([train_df_prepared, train_df_label_duration], axis=1)
temp_df['Group_Key'] = train_df_data['MAC Address'].values 

def create_grouped_sequences(df, feature_cols, target_col, time_steps=12):
    X, y = [], []
    
    # Group by the original string/ID, not the encoded bits
    for mac, group in df.groupby('Group_Key'):
        # Convert to float32
        group_features = group[feature_cols].values.astype('float32')
        group_target = group[target_col].values.astype('float32')
        
        if len(group_features) > time_steps:
            for i in range(len(group_features) - time_steps):
                X.append(group_features[i : (i + time_steps)])
                y.append(group_target[i + time_steps])
    
    return np.array(X), np.array(y)

# 3. Generate sequences
X_final, y_final = create_grouped_sequences(
    temp_df, 
    feature_cols=features, 
    target_col='Total_Duration_Secs', # Ensure this matches your label column name
    time_steps=12
)

In [37]:
final_test_df_data

,mac_bits__MAC Address_0,mac_bits__MAC Address_1,mac_bits__MAC Address_2,mac_bits__MAC Address_3,mac_bits__MAC Address_4,mac_bits__MAC Address_5,mac_bits__MAC Address_6,std_scaler__temp,std_scaler__precip,std_scaler__clouds,...,std_scaler__Hour,std_scaler__DayOfWeek,std_scaler__Month,std_scaler__IsWeekend,std_scaler__IsHoliday,std_scaler__Hour_Sin,std_scaler__Hour_Cos,std_scaler__Month_Sin,std_scaler__Month_Cos,std_scaler__Plug Type Encoded
207532,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207533,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207534,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207535,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
207536,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.921998,-0.178408,-1.069321,...,0.187263,1.123632,-0.371793,1.803477,-0.135783,-0.508161,-0.808299,0.635725,-1.237721,-0.151759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259410,0.0,1.0,1.0,1.0,0.0,0.0,1.0,-0.584554,-0.178408,0.132092,...,0.638334,0.080924,1.651667,-0.554484,-0.135783,-1.018474,-0.096084,-0.065080,1.431270,-0.151759
259411,0.0,1.0,1.0,0.0,0.0,1.0,0.0,-0.584554,-0.178408,0.132092,...,0.638334,0.080924,1.651667,-0.554484,-0.135783,-1.018474,-0.096084,-0.065080,1.431270,-0.151759
259412,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.601871,-0.178408,0.132092,...,0.863869,0.080924,1.651667,-0.554484,-0.135783,-1.157755,0.373207,-0.065080,1.431270,-0.151759
259413,0.0,1.0,1.0,0.0,0.0,1.0,0.0,-0.601871,-0.178408,0.132092,...,0.863869,0.080924,1.651667,-0.554484,-0.135783,-1.157755,0.373207,-0.065080,1.431270,-0.151759


In [45]:
# 1. Prep the test data similarly to the train data
features = [col for col in final_test_df_data.columns]

temp_test_df = pd.concat([final_test_df_data, test_df_label_duration], axis=1)
temp_test_df['Group_Key'] = test_df_data['MAC Address'].values

# 2. Use your existing function to generate the test sequences
X_test_seq, y_test_seq = create_grouped_sequences(
    temp_test_df, 
    feature_cols=features, 
    target_col='Total_Duration_Secs', 
    time_steps=12
)

In [46]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

def build_lstm_regressor(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.30),
        LSTM(64),
        BatchNormalization(),
        Dropout(0.25),
        Dense(64, activation='relu'),
        Dropout(0.15),
        Dense(1, activation='linear'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=Huber(delta=1.0),
        metrics=['mae'],
    )
    return model


lstm_model = build_lstm_regressor((X_final.shape[1], X_final.shape[2]))
lstm_model.summary()

d:\University\Final_Project\Coding\AI\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 12, 128)        │        76,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 12, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 131,201 (512.50 KB)

 Trainable params: 130,817 (511.00 KB)

 Non-trainable params: 384 (1.50 KB)

In [47]:
from pathlib import Path
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

lstm_model_path = Path(config.model_dir) / 'lstm_duration.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    ModelCheckpoint(lstm_model_path, monitor='val_loss', save_best_only=True),
]

history = lstm_model.fit(
    X_final, 
    y_final,
    validation_data=(X_test_seq, y_test_seq),
    epochs=15,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 56s 16ms/step - loss: 5700.9741 - mae: 5701.4805 - val_loss: 4401.7173 - val_mae: 4402.2173 - learning_rate: 0.0010
Epoch 2/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 92s 29ms/step - loss: 4957.2388 - mae: 4957.7310 - val_loss: 4374.7363 - val_mae: 4375.2363 - learning_rate: 0.0010
Epoch 3/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 64s 20ms/step - loss: 4955.6040 - mae: 4956.0913 - val_loss: 4377.2563 - val_mae: 4377.7563 - learning_rate: 0.0010
Epoch 4/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 64s 20ms/step - loss: 4941.3267 - mae: 4941.8335 - val_loss: 4365.9106 - val_mae: 4366.4106 - learning_rate: 0.0010
Epoch 5/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 63s 19ms/step - loss: 4942.5708 - mae: 4943.0679 - val_loss: 4357.7817 - val_mae: 4358.2817 - learning_rate: 0.0010
Epoch 6/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 96s 30ms/step - loss: 4927.2749 - mae: 4927.7681 - val_loss: 4359.0234 - val_mae: 4359.5234 - learning_rate: 0.0010
Epoch 7/15
3231/3231 ━━━━━━━━━━━━━━━━━━━━ 82s 25ms/s

In [49]:
joblib.dump(lstm_model, 'model_B/lstm.keras')

['model_B/lstm.keras']